<a href="https://colab.research.google.com/github/yohperez/RobotPaseadorPerro/blob/main/Robot_paseador_mejorado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 BotPro — Robot Paseador de Perros en Madrid

**Simulación completa del servicio:** registro de usuario, reserva de paseo, configuración del robot, paseo con eventos dinámicos, devolución segura y pago final.

---

### 🗂️ Estructura del notebook

| Celda | Contenido |
|---|---|
| 1 | Importaciones y utilidades |
| 2 | Clase `RobotPaseador` |
| 3 | Clase `AppPaseo` |
| 4 | Simulación completa del servicio |
| 5 | Demo rápida (sin inputs manuales) |

> **Consejo:** Ejecuta las celdas en orden con `Shift+Enter` o usa *Runtime → Run all*.

## 📦 Celda 1 — Importaciones y utilidades

In [1]:
import math
import time
import random

# ─────────────────────────────────────────────
# Función auxiliar: separador visual de sección
# ─────────────────────────────────────────────
def seccion(titulo: str, emoji: str = "─") -> None:
    """Imprime un separador con título para mayor legibilidad en la consola."""
    linea = emoji * 3
    print(f"\n{linea} {titulo} {linea}")

# ─────────────────────────────────────────────
# Función auxiliar: validar input con opciones
# ─────────────────────────────────────────────
def pedir_opcion(prompt: str, opciones_validas: list, transform=str.lower) -> str:
    """
    Solicita al usuario una opción hasta que introduzca una válida.
    Evita que el programa se rompa con entradas inesperadas.
    """
    while True:
        respuesta = transform(input(prompt).strip())
        if respuesta in opciones_validas:
            return respuesta
        print(f"  ⚠️  Opción no válida. Elige entre: {', '.join(opciones_validas)}")

print("✅ Celda 1 cargada — Importaciones y utilidades listas.")

✅ Celda 1 cargada — Importaciones y utilidades listas.


## 🤖 Celda 2 — Clase `RobotPaseador`

In [2]:
class RobotPaseador:
    """
    Representa al robot físico BotPro.
    Gestiona batería, equipamiento, normativa y eventos durante el paseo.
    """

    # Coordenadas GPS de Madrid (aprox.)
    ORIGEN_GPS   = (40.4110, -3.7092)  # Centro Deportivo La Cebada
    MAPA_DESTINOS = {
        "retiro":      (40.4153, -3.6835),
        "plaza mayor": (40.4153, -3.7073),
        "madrid río":  (40.4121, -3.7210),
    }

    def __init__(self, modelo: str = "BotPro v2.1", bateria: int = 45):
        self.modelo            = modelo
        self.bateria           = bateria          # % de carga
        self.deposito_bolsas   = 0                # bolsas usadas
        self.num_bolsas        = 8                # bolsas disponibles
        self.normas_madrid     = []
        self.nombre_perro      = ""
        self.destino           = ""

        self.equipamiento = {
            "correa_corta":           True,
            "chip_identificativo":    True,
            "seguro_rc":              True,
            "botella_agua_limpieza":  True,
        }

    # ── CONFIGURACIÓN ─────────────────────────────────────────────

    def configurar_sistema(self) -> bool:
        """Verifica batería, equipamiento y descarga normativa. Devuelve True si todo OK."""
        seccion(f"⚙️  Configuración {self.modelo}")

        # Gestión de batería
        if self.bateria < 50:
            print(f"⚠️  Batería al {self.bateria}%. Realizando carga rápida... ", end="")
            self.bateria = 100
            print("¡100% listo!")
        else:
            print(f"✅ Batería al {self.bateria}%. Nivel óptimo.")

        # Verificación de equipamiento
        faltantes = [item for item, estado in self.equipamiento.items() if not estado]
        if faltantes:
            print(f"❌ Faltan elementos de equipamiento: {faltantes}")
            return False
        if self.num_bolsas < 1:
            print("❌ Sin bolsas de limpieza. Reposición necesaria.")
            return False

        # Descarga de normativa
        print("📥 Sincronizando normativa municipal de Madrid...")
        self.normas_madrid = [
            "Caminar por la acera, lejos de la calzada.",
            "Cruzar solo en pasos de cebra con semáforo verde.",
            "Uso obligatorio de correa corta (máx. 2 m).",
            "Recoger excrementos y depositarlos en contenedor.",
            "Diluir orina con botella de agua (Ordenanza Municipal).",
        ]
        print("✅ Normativa cargada. Rutas sincronizadas. Todo en orden.")
        return True

    def mostrar_normas(self) -> None:
        """Imprime las normas de tránsito cargadas."""
        seccion("🚦 Normativa DGT y Ordenanza de Madrid")
        for i, norma in enumerate(self.normas_madrid, 1):
            print(f"   {i}. {norma}")

    def planificar_ruta(self) -> None:
        """Calcula la distancia aproximada al destino usando distancia euclidiana."""
        destino_key = self.destino.lower()
        if destino_key in self.MAPA_DESTINOS:
            x1, y1 = self.ORIGEN_GPS
            x2, y2 = self.MAPA_DESTINOS[destino_key]
            # 1 grado ≈ 111 km
            distancia_km = math.sqrt((x2 - x1)**2 + (y2 - y1)**2) * 111
            print(f"📍 Ruta: La Cebada (La Latina) ➔ {self.destino.title()}")
            print(f"📏 Distancia estimada: {distancia_km:.2f} km")
        else:
            print(f"📍 Ruta libre por La Latina (destino '{self.destino}' no mapeado).")

    # ── EVENTOS DURANTE EL PASEO ───────────────────────────────────

    def caminar(self) -> None:
        """Simula la marcha por la acera."""
        print(f"🚶 Caminando por la acera. {self.nombre_perro} va en correa corta.")
        print("🛑 Semáforo rojo en Tirso de Molina. Esperando...")
        time.sleep(1)
        print("🟢 Verde. Cruzando el paso de cebra con cuidado.")

    def hidratar_mascota(self, minuto: int) -> None:
        """Pausa de hidratación cada N minutos y simula notificación a la app."""
        print(f"\n💧 [{minuto} min] Deteniendo para que {self.nombre_perro} beba agua fresca.")
        print(f"📸 Notificación enviada: foto de {self.nombre_perro} feliz → App del dueño.")

    def reaccionar_a_entorno(self, evento: str) -> None:
        """
        Gestiona eventos del entorno.
        Eventos válidos: 'charco', 'perro_defeca', 'pis', 'lluvia'.
        """
        eventos_validos = {"charco", "perro_defeca", "pis", "lluvia"}
        if evento not in eventos_validos:
            print(f"⚠️  Evento desconocido: '{evento}'")
            return

        if evento == "charco":
            print("🌊 Charco detectado. Recalculando paso para evitar que se moje las patitas.")

        elif evento == "perro_defeca":
            print(f"💩 {self.nombre_perro} ha defecado. Iniciando protocolo de limpieza...")
            if self.num_bolsas > 0:
                print("   → Saco bolsa, recojo, cierro, guardo en depósito. ¡Ciudad limpia!")
                self.num_bolsas      -= 1
                self.deposito_bolsas += 1
            else:
                print("   ⚠️  Sin bolsas disponibles. Buscando dispensador público más cercano.")

        elif evento == "pis":
            print("💦 Aplicando agua de la botella según Ordenanza Municipal. ¡Impecable!")

        elif evento == "lluvia":
            print("🌧️  Lluvia detectada. Buscando refugio seco para la mascota...")
            time.sleep(1)
            print("☀️  Lluvia finalizada. Reanudando la ruta.")

    # ── POST-SERVICIO ──────────────────────────────────────────────

    def protocolo_cierre(self) -> None:
        """Limpieza y recarga para el siguiente servicio."""
        seccion("🔧 Mantenimiento en Central")
        print(f"♻️  Vaciando depósito: {self.deposito_bolsas} desecho(s) procesado(s) de forma ecológica.")
        self.bateria = 100
        print(f"🔋 Batería recargada al 100%. Bolsas restantes: {self.num_bolsas}.")
        print("✅ Robot limpio y listo para el siguiente servicio.")


print("✅ Celda 2 cargada — Clase RobotPaseador definida.")

✅ Celda 2 cargada — Clase RobotPaseador definida.


## 📱 Celda 3 — Clase `AppPaseo`

In [3]:
class AppPaseo:
    """
    Gestiona el registro de usuario, reserva del paseo y facturación.
    """

    PLANES = {
        "bronce": {"min": 15,  "mtrs": 350,  "precio": 15.0},
        "plata":  {"min": 30,  "mtrs": 750,  "precio": 30.0},
        "oro":    {"min": 60,  "mtrs": 1000, "precio": 45.0},
    }
    PRECIO_EXTRA_POR_MIN = 0.50  # €/minuto adicional

    # Códigos de seguridad (en producción llegarían de un servidor)
    CODIGO_RECOGIDA  = "7744"
    CODIGO_ENTREGA   = "1234"

    def registrar_usuario(self) -> dict:
        """Recoge los datos del dueño. Valida que no queden campos vacíos."""
        seccion("📱 App BotPro — Registro de usuario")

        email      = input("📧 Email: ").strip()
        _          = input("🔒 Contraseña: ").strip()   # no se almacena
        nombre     = input("👤 Nombre del dueño: ").strip().title()
        apellido   = input("👤 Apellido: ").strip().title()
        dni        = input("🪪 DNI: ").strip().upper()
        rgpd       = pedir_opcion("¿Acepta la política de protección de datos? (s/n): ", ["s", "n"])

        if rgpd == "n":
            print("❌ Registro cancelado. Debe aceptar la política para continuar.")
            raise SystemExit(0)

        print(f"\n✅ Registro completado. Bienvenido/a, {nombre} {apellido}.")
        return {"nombre": nombre, "apellido": apellido, "email": email, "dni": dni}

    def gestionar_reserva(self) -> dict:
        """Recoge los datos de la mascota y el plan elegido. Devuelve la reserva completa."""
        # ── Registro del dueño ──
        dueño = self.registrar_usuario()

        # ── Datos de la mascota ──
        seccion("📱 App BotPro — Reserva de paseo")
        mascota   = input("🐶 Nombre de la mascota: ").strip().title()
        chip      = input("💾 Código de microchip: ").strip()
        fecha     = input("📅 Fecha del paseo (dd/mm/aaaa): ").strip()
        hora      = input("🕐 Hora de recogida (hh:mm): ").strip()
        tamano    = pedir_opcion(
            "📏 Tamaño del perro (grande/mediano/pequeño): ",
            ["grande", "mediano", "pequeño"]
        )

        # ── Elección del plan ──
        print("\n💼 Planes disponibles:")
        for nombre_plan, datos in self.PLANES.items():
            print(f"   • {nombre_plan.capitalize():8s} → {datos['min']} min | {datos['mtrs']} m | {datos['precio']:.0f}€")

        plan_sel = pedir_opcion("Seleccione plan: ", list(self.PLANES.keys()))
        total    = self.PLANES[plan_sel]["precio"]
        minutos  = self.PLANES[plan_sel]["min"]

        # ── Tiempo extra ──
        extra = pedir_opcion("⏱️  ¿Deseas tiempo extra? (0.50€/min) (si/no): ", ["si", "no"])
        if extra == "si":
            while True:
                try:
                    mins_extra = int(input("   Minutos adicionales: ").strip())
                    if mins_extra > 0:
                        break
                    print("   ⚠️  Introduce un número positivo.")
                except ValueError:
                    print("   ⚠️  Introduce un número entero.")
            total   += mins_extra * self.PRECIO_EXTRA_POR_MIN
            minutos += mins_extra

        # ── Datos de entrega ──
        indicaciones = input("📝 Indicaciones especiales (Enter si no hay): ").strip()
        domicilio    = input("🏠 Domicilio de recogida: ").strip()

        # ── Destino del paseo ──
        destinos_disponibles = ["retiro", "plaza mayor", "madrid río", "libre"]
        print(f"\n🗺️  Destinos disponibles: {', '.join(d.title() for d in destinos_disponibles)}")
        destino = pedir_opcion("¿A dónde iremos hoy?: ", destinos_disponibles)

        print(f"\n🧾 Resumen: {mascota} | Plan {plan_sel.capitalize()} | {minutos} min | Total: {total:.2f}€")

        return {
            "dueño":            dueño["nombre"],
            "mascota":          mascota,
            "total":            total,
            "minutos":          minutos,
            "domicilio":        domicilio,
            "destino":          destino,
            "indicaciones":     indicaciones,
            "codigo_recogida":  self.CODIGO_RECOGIDA,
            "codigo_entrega":   self.CODIGO_ENTREGA,
        }

    def procesar_pago_y_feedback(self, reserva: dict, robot: 'RobotPaseador') -> None:
        """Solicita pago y valoración del servicio."""
        seccion("💳 Pago y Calificación")

        print(f"💰 Total a pagar: {reserva['total']:.2f}€")
        metodo = pedir_opcion(
            "💳 Método de pago (tarjeta/bizum/efectivo): ",
            ["tarjeta", "bizum", "efectivo"]
        )
        propina_str = input("🎁 ¿Propina para mantenimiento? (0 si no deseas): ").strip()
        try:
            propina = float(propina_str) if propina_str else 0.0
        except ValueError:
            propina = 0.0

        total_final = reserva['total'] + propina
        print(f"✅ Pago de {total_final:.2f}€ procesado con {metodo}. ¡Gracias!")

        # Calificación
        print("\n⭐ Valoración del servicio (1–10)")
        satisfaccion = input("   Satisfacción general: ").strip()
        puntualidad  = input("   Puntualidad: ").strip()
        sugerencia   = pedir_opcion("¿Tienes alguna sugerencia? (si/no): ", ["si", "no"])
        if sugerencia == "si":
            comentario = input("   Cuéntanos: ").strip()
            print(f"   📩 Sugerencia registrada: '{comentario}'")

        print(f"\n🎉 {reserva['dueño']}, ¡gracias por confiar en BotPro!")
        print(f"   {reserva['mascota']} y yo te deseamos un día fantástico. ¡Hasta pronto!")


print("✅ Celda 3 cargada — Clase AppPaseo definida.")

✅ Celda 3 cargada — Clase AppPaseo definida.


## 🚀 Celda 4 — Simulación completa del servicio

> Esta celda pide inputs al usuario. Tendrás que rellenar los campos en la caja de texto que aparece bajo la celda.

In [4]:
# ════════════════════════════════════════════════
#   SIMULACIÓN COMPLETA — BotPro Robot Paseador
# ════════════════════════════════════════════════

# ── 1. RESERVA ────────────────────────────────────────────────────
app    = AppPaseo()
reserva = app.gestionar_reserva()

# ── 2. CONFIGURACIÓN DEL ROBOT ────────────────────────────────────
robot = RobotPaseador("BotPro v2.1", bateria=45)
robot.nombre_perro = reserva["mascota"]
robot.destino      = reserva["destino"]

listo = robot.configurar_sistema()
if not listo:
    raise SystemExit("❌ Robot no preparado. Revisa el equipamiento.")

robot.mostrar_normas()
robot.planificar_ruta()

# ── 3. INICIO DEL SERVICIO ────────────────────────────────────────
seccion("🚀 Inicio del Servicio")
print(f"🔔 Robot llega a {reserva['domicilio']}. *Toca el timbre*")
print(f"🤖 '¡Hola, {reserva['dueño']}! He venido a buscar a {reserva['mascota']}.✨'")

# Verificación código de recogida
intentos = 0
while intentos < 3:
    cod = input(f"🔑 Introduce el código de recogida ({reserva['codigo_recogida']}): ").strip()
    if cod == reserva['codigo_recogida']:
        print("✅ ¡Código correcto! Arnés desbloqueado. ¡Listos para salir!")
        break
    intentos += 1
    restantes = 3 - intentos
    if restantes > 0:
        print(f"❌ Código incorrecto. Te quedan {restantes} intento(s).")
    else:
        raise SystemExit("🚫 Demasiados intentos fallidos. Servicio cancelado por seguridad.")

input("📱 Confirma el inicio del paseo en la App (pulsa Enter): ")

# ── 4. EL PASEO ───────────────────────────────────────────────────
seccion("🐕 En el Paseo")
robot.caminar()
robot.reaccionar_a_entorno("charco")

# Hidratación cada 10 minutos
for m in range(10, reserva['minutos'] + 1, 10):
    robot.hidratar_mascota(m)

# Eventos aleatorios del paseo
robot.reaccionar_a_entorno("perro_defeca")
robot.reaccionar_a_entorno("pis")
robot.reaccionar_a_entorno("lluvia")

print(f"\n🏁 '{reserva['mascota']} se ha portado genial. ¡Volvemos a casa!'")

# ── 5. DEVOLUCIÓN Y CÓDIGO DE ENTREGA ────────────────────────────
seccion("🏠 Devolución")
print(f"🔔 Robot llega al domicilio. El arnés permanece BLOQUEADO por seguridad.")

intentos = 0
while intentos < 3:
    cod_fin = input(f"🔑 Introduce el código de entrega ({reserva['codigo_entrega']}): ").strip()
    if cod_fin == reserva['codigo_entrega']:
        print(f"✅ ¡Arnés liberado! Ha sido un placer cuidar de {reserva['mascota']}.")
        break
    intentos += 1
    restantes = 3 - intentos
    if restantes > 0:
        print(f"❌ Código incorrecto. Te quedan {restantes} intento(s).")
    else:
        raise SystemExit("🚫 Código de entrega incorrecto. Llamando a soporte técnico.")

input("📱 Confirma la recepción en la App (pulsa Enter): ")
print("🤖 '¡Adiós! Espero que nos veamos pronto para otra aventura.'")

# ── 6. POST-SERVICIO ─────────────────────────────────────────────
robot.protocolo_cierre()

# ── 7. PAGO Y FEEDBACK ───────────────────────────────────────────
app.procesar_pago_y_feedback(reserva, robot)


─── 📱 App BotPro — Registro de usuario ───
📧 Email: coco@chanel
🔒 Contraseña: 12345678
👤 Nombre del dueño: coco
👤 Apellido: chanel
🪪 DNI: 12345678A
¿Acepta la política de protección de datos? (s/n): s

✅ Registro completado. Bienvenido/a, Coco Chanel.

─── 📱 App BotPro — Reserva de paseo ───
🐶 Nombre de la mascota: Coco
💾 Código de microchip: 1234
📅 Fecha del paseo (dd/mm/aaaa): 17/03/26
🕐 Hora de recogida (hh:mm): 12
📏 Tamaño del perro (grande/mediano/pequeño): grande

💼 Planes disponibles:
   • Bronce   → 15 min | 350 m | 15€
   • Plata    → 30 min | 750 m | 30€
   • Oro      → 60 min | 1000 m | 45€
Seleccione plan: Oro
⏱️  ¿Deseas tiempo extra? (0.50€/min) (si/no): si
   Minutos adicionales: no
   ⚠️  Introduce un número entero.
   Minutos adicionales: 0
   ⚠️  Introduce un número positivo.
   Minutos adicionales: 1
📝 Indicaciones especiales (Enter si no hay): 
🏠 Domicilio de recogida: coco 16

🗺️  Destinos disponibles: Retiro, Plaza Mayor, Madrid Río, Libre
¿A dónde iremos hoy?: R

## ⚡ Celda 5 — Demo rápida (sin inputs manuales)

Ejecuta el flujo completo con **datos precargados** para ver todos los mensajes sin necesidad de escribir nada. Útil para hacer una demostración rápida o probar el código.

In [5]:
# ════════════════════════════════════════════════
#   DEMO RÁPIDA — Datos precargados, sin inputs
# ════════════════════════════════════════════════

# Reserva simulada (equivalente a rellenar el formulario con estos datos)
reserva_demo = {
    "dueño":           "Mariajose",
    "mascota":         "Otto",
    "total":           52.50,
    "minutos":         75,       # Plan Oro (60 min) + 15 min extra
    "domicilio":       "Calle Juan José, 4 — La Latina",
    "destino":         "madrid río",
    "indicaciones":    "Otto es muy juguetero con otros perros.",
    "codigo_recogida": "7744",
    "codigo_entrega":  "1234",
}

# Instanciamos y configuramos el robot
robot_demo = RobotPaseador("BotPro v2.1", bateria=45)
robot_demo.nombre_perro = reserva_demo["mascota"]
robot_demo.destino      = reserva_demo["destino"]

listo = robot_demo.configurar_sistema()
if not listo:
    raise SystemExit("❌ Robot no preparado.")

robot_demo.mostrar_normas()
robot_demo.planificar_ruta()

# Inicio del servicio
seccion("🚀 Inicio del Servicio (Demo)")
print(f"🔔 Robot llega a '{reserva_demo['domicilio']}'")
print(f"🤖 '¡Hola, {reserva_demo['dueño']}! He venido a buscar a {reserva_demo['mascota']}.'")
print(f"✅ Código de recogida verificado: {reserva_demo['codigo_recogida']}. Arnés desbloqueado.")
print("📱 Inicio de paseo confirmado en App.")

# El paseo
seccion("🐕 En el Paseo (Demo)")
robot_demo.caminar()
robot_demo.reaccionar_a_entorno("charco")

for m in range(10, reserva_demo['minutos'] + 1, 10):
    robot_demo.hidratar_mascota(m)

robot_demo.reaccionar_a_entorno("perro_defeca")
robot_demo.reaccionar_a_entorno("pis")
robot_demo.reaccionar_a_entorno("lluvia")
print(f"\n🏁 '{reserva_demo['mascota']} se ha portado genial. ¡Volvemos a casa!'")

# Devolución
seccion("🏠 Devolución (Demo)")
print(f"✅ Código de entrega verificado: {reserva_demo['codigo_entrega']}. Arnés liberado.")
print(f"✅ Recepción de {reserva_demo['mascota']} confirmada en App.")
print("🤖 '¡Adiós! ¡Hasta la próxima aventura!'")

# Post-servicio
robot_demo.protocolo_cierre()

# Pago
seccion("💳 Pago y Calificación (Demo)")
print(f"💰 Total: {reserva_demo['total']:.2f}€ | Método: Tarjeta | Propina: 5.00€")
print("✅ Pago de 57.50€ procesado. ⭐⭐⭐⭐⭐ (10/10 en satisfacción y puntualidad)")
print(f"\n🎉 {reserva_demo['dueño']}, ¡gracias por confiar en BotPro!")
print(f"   {reserva_demo['mascota']} y yo te deseamos un día fantástico. 🐾")


─── ⚙️  Configuración BotPro v2.1 ───
⚠️  Batería al 45%. Realizando carga rápida... ¡100% listo!
📥 Sincronizando normativa municipal de Madrid...
✅ Normativa cargada. Rutas sincronizadas. Todo en orden.

─── 🚦 Normativa DGT y Ordenanza de Madrid ───
   1. Caminar por la acera, lejos de la calzada.
   2. Cruzar solo en pasos de cebra con semáforo verde.
   3. Uso obligatorio de correa corta (máx. 2 m).
   4. Recoger excrementos y depositarlos en contenedor.
   5. Diluir orina con botella de agua (Ordenanza Municipal).
📍 Ruta: La Cebada (La Latina) ➔ Madrid Río
📏 Distancia estimada: 1.32 km

─── 🚀 Inicio del Servicio (Demo) ───
🔔 Robot llega a 'Calle Juan José, 4 — La Latina'
🤖 '¡Hola, Mariajose! He venido a buscar a Otto.'
✅ Código de recogida verificado: 7744. Arnés desbloqueado.
📱 Inicio de paseo confirmado en App.

─── 🐕 En el Paseo (Demo) ───
🚶 Caminando por la acera. Otto va en correa corta.
🛑 Semáforo rojo en Tirso de Molina. Esperando...
🟢 Verde. Cruzando el paso de cebra con c